**Origem:** Silver

**Objetivo:** Aplicar regras de negócio para geração de insights analíticos sobre o mercado de criptoativos, produzindo métricas e classificações prontas para consumo por dashboards e times de análise.

---
### Regras de Negócio Aplicadas

| # | Coluna Gold | Descrição |
|---|-------------|-----------|
| 1 | `categoria_market_cap` | Classificação por tamanho de mercado |
| 2 | `score_liquidez` | Score de liquidez relativa (volume/market cap) |
| 3 | `categoria_liquidez` | Classificação qualitativa da liquidez |
| 4 | `tendencia_curto_prazo` | Sinal de tendência em 24h |
| 5 | `tendencia_medio_prazo` | Sinal de tendência em 7d |
| 6 | `sinal_mercado` | Sinal composto (curto + médio prazo) |
| 7 | `categoria_volatilidade` | Classificação de volatilidade (24h) |
| 8 | `distancia_ath_categoria` | Distância do All-Time High categorizada |
| 9 | `flag_supply_escasso` | Flag de ativo com supply próximo do máximo |
| 10 | `flag_stablecoin` | Identificação de stablecoins |
| 11 | `score_risco` | Score de risco composto (0–10) |
| 12 | `categoria_risco` | Classificação de risco final |
| 13 | `oportunidade_compra` | Flag de potencial oportunidade de compra |
| 14 | `preco_brl` | Preço convertido para BRL (câmbio do dia) |
| 15 | `market_cap_brl` | Market cap em BRL |
| 16 | `ranking_volume_na_categoria` | Rank de volume dentro de cada faixa de cap |
| 17 | `processado_gold_em` | Timestamp de auditoria da camada Gold |

##Configuração e Leitura da Silver

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
from delta.tables import DeltaTable

spark = SparkSession.builder.appName("Gold_Crypto").getOrCreate()

CATALOG = "projetos"
SCHEMA  = "crypto"

# Silver já existente
TABELA_SILVER = f"{CATALOG}.{SCHEMA}.silver_market_data"

# Gold será criada
TABELA_GOLD = f"{CATALOG}.{SCHEMA}.gold_market_data"

# Leitura direta da tabela Silver
df_silver = spark.table(TABELA_SILVER)

##Regras de Negócio

###Categoria de Market Cap
Classifica o ativo pelo tamanho de mercado, seguindo a convenção do setor:
- **Large Cap** → > US$ 10 bi
- **Mid Cap**   → US$ 1 bi – US$ 10 bi
- **Small Cap** → US$ 500 mi – US$ 1 bi
- **Micro Cap** → < US$ 500 mi

In [0]:
df_gold = df_silver.withColumn(
    "categoria_market_cap",
    F.when(F.col("market_cap_usd") > 10_000_000_000, "Large Cap")
     .when(F.col("market_cap_usd") >  1_000_000_000, "Mid Cap")
     .when(F.col("market_cap_usd") >    500_000_000, "Small Cap")
     .otherwise("Micro Cap")
)

###Score e Categoria de Liquidez
**Score de Liquidez** = Volume 24h / Market Cap (turnover diário).
- **Alta Liquidez**  → Score > 0.20
- **Média Liquidez** → Score > 0.05
- **Baixa Liquidez** → Score ≤ 0.05

In [0]:
df_gold = df_gold.withColumn(
    "score_liquidez",
    F.round(F.col("volume_24h_usd") / F.col("market_cap_usd"), 4)
).withColumn(
    "categoria_liquidez",
    F.when(F.col("score_liquidez") > 0.20, "Alta Liquidez")
     .when(F.col("score_liquidez") > 0.05, "Média Liquidez")
     .otherwise("Baixa Liquidez")
)

###Tendências de Curto e Médio Prazo
| Sinal | Critério 24h | Critério 7d |
|-------|-------------|------------|
| Bullish | > +1% | > +3% |
| Bearish | < -1% | < -3% |
| Lateral | entre -1% e +1% | entre -3% e +3% |

In [0]:
df_gold = df_gold.withColumn(
    "tendencia_curto_prazo",
    F.when(F.col("variacao_pct_24h") >  1.0, "Bullish")
     .when(F.col("variacao_pct_24h") < -1.0, "Bearish")
     .otherwise("Lateral")
).withColumn(
    "tendencia_medio_prazo",
    F.when(F.col("variacao_pct_7d") >  3.0, "Bullish")
     .when(F.col("variacao_pct_7d") < -3.0, "Bearish")
     .otherwise("Lateral")
).withColumn(
    "sinal_mercado",
    F.when(
        (F.col("tendencia_curto_prazo") == "Bullish") &
        (F.col("tendencia_medio_prazo") == "Bullish"), "Forte Alta"
    ).when(
        (F.col("tendencia_curto_prazo") == "Bearish") &
        (F.col("tendencia_medio_prazo") == "Bearish"), "Forte Queda"
    ).when(
        (F.col("tendencia_curto_prazo") == "Bullish") &
        (F.col("tendencia_medio_prazo") != "Bullish"), "Alta de Curto Prazo"
    ).when(
        (F.col("tendencia_curto_prazo") == "Bearish") &
        (F.col("tendencia_medio_prazo") != "Bearish"), "Queda de Curto Prazo"
    ).otherwise("Consolidação")
)

###Categoria de Volatilidade
Baseada no módulo da variação em 24h:
- **Extremamente Volátil** → |Δ24h| > 10%
- **Alta Volatilidade**   → |Δ24h| > 5%
- **Moderada**            → |Δ24h| > 2%
- **Baixa Volatilidade**  → |Δ24h| ≤ 2%

In [0]:
df_gold = df_gold.withColumn(
    "variacao_abs_24h", F.abs(F.col("variacao_pct_24h"))
).withColumn(
    "categoria_volatilidade",
    F.when(F.col("variacao_abs_24h") > 10.0, "Extremamente Volátil")
     .when(F.col("variacao_abs_24h") >  5.0, "Alta Volatilidade")
     .when(F.col("variacao_abs_24h") >  2.0, "Moderada")
     .otherwise("Baixa Volatilidade")
).drop("variacao_abs_24h")

###Distância do All-Time High (ATH)
- **Próximo do ATH**     → queda < 20% vs ATH
- **Desconto Moderado** → queda entre 20% e 50%
- **Desconto Alto**     → queda entre 50% e 80%
- **Bear Market Deep**  → queda > 80%

In [0]:
df_gold = df_gold.withColumn(
    "distancia_ath_categoria",
    F.when(F.col("variacao_pct_vs_ath") >= -20.0, "Próximo do ATH")
     .when(F.col("variacao_pct_vs_ath") >= -50.0, "Desconto Moderado")
     .when(F.col("variacao_pct_vs_ath") >= -80.0, "Desconto Alto")
     .otherwise("Bear Market Deep")
)

###Flag de Stablecoin
Identifica stablecoins pelo preço próximo de US$ 1.00 (tolerância ±2%).

In [0]:
df_gold = df_gold.withColumn(
    "flag_stablecoin",
    F.when(
        (F.col("preco_usd") >= 0.98) & (F.col("preco_usd") <= 1.02), True
    ).otherwise(False)
)

###Score de Risco Composto (0–10)
Pontuação aditiva — quanto maior, maior o risco:

| Componente | Regra | Pontos |
|-----------|-------|--------|
| Volatilidade extrema | variacao abs 24h > 10% | +4 |
| Alta volatilidade    | variacao abs 24h > 5%  | +2 |
| Micro Cap            | Market cap < 500 mi | +2 |
| Small Cap            | Market cap < 1 bi   | +1 |
| Baixa liquidez       | Score liquidez ≤ 0.05 | +1 |
| Não é stablecoin     | Preço diferente de ~$1 | +1 |
| Bear Market Deep     | Queda > 80% do ATH | +1 |

In [0]:
df_gold = df_gold.withColumn(
    "score_risco",
    F.least(
        F.lit(10.0),
        F.when(F.abs(F.col("variacao_pct_24h")) > 10.0, 4.0)
         .when(F.abs(F.col("variacao_pct_24h")) >  5.0, 2.0)
         .otherwise(0.0)
        + F.when(F.col("market_cap_usd") < 500_000_000,   2.0)
           .when(F.col("market_cap_usd") < 1_000_000_000, 1.0)
           .otherwise(0.0)
        + F.when(F.col("score_liquidez") <= 0.05, 1.0).otherwise(0.0)
        + F.when(F.col("flag_stablecoin") == False, 1.0).otherwise(0.0)
        + F.when(F.col("variacao_pct_vs_ath") < -80.0, 1.0).otherwise(0.0)
    )
).withColumn(
    "categoria_risco",
    F.when(F.col("score_risco") >= 7.0, "Muito Alto")
     .when(F.col("score_risco") >= 5.0, "Alto")
     .when(F.col("score_risco") >= 3.0, "Moderado")
     .when(F.col("score_risco") >= 1.0, "Baixo")
     .otherwise("Mínimo")
)

###Flag de Oportunidade de Compra
Combina múltiplos critérios:
- Não é stablecoin
- Desconto > 50% vs ATH
- Tendência de 7d não está em queda forte
- Liquidez média ou alta
- Não é Micro Cap

In [0]:
df_gold = df_gold.withColumn(
    "oportunidade_compra",
    F.when(
        (F.col("flag_stablecoin") == False) &
        (F.col("variacao_pct_vs_ath") < -50.0) &
        (F.col("tendencia_medio_prazo") != "Bearish") &
        (F.col("categoria_liquidez").isin("Alta Liquidez", "Média Liquidez")) &
        (F.col("categoria_market_cap") != "Micro Cap"),
        True
    ).otherwise(False)
)

###Ranking por Categoria de Market Cap
Ranking de volume 24h dentro de cada categoria.

In [0]:
w_rank = Window.partitionBy("categoria_market_cap").orderBy(F.desc("volume_24h_usd"))

df_gold = df_gold.withColumn(
    "ranking_volume_na_categoria",
    F.rank().over(w_rank)
)

###Coluna de Auditoria

In [0]:
df_gold = df_gold.withColumn(
    "processado_gold_em",
    F.current_timestamp()
)

##Validação e Preview

In [0]:
colunas_gold = [
    "coin_id", "symbol", "name", "preco_usd",
    "market_cap_usd", "categoria_market_cap",
    "score_liquidez", "categoria_liquidez",
    "tendencia_curto_prazo", "tendencia_medio_prazo", "sinal_mercado",
    "categoria_volatilidade", "distancia_ath_categoria",
    "flag_stablecoin", "score_risco", "categoria_risco",
    "oportunidade_compra", "ranking_volume_na_categoria",
    "data_arquivo", "processado_gold_em", "extraido_em"
]

df_gold_final = df_gold.select(colunas_gold)

print(f"Gold gerada: {df_gold_final.count()} linhas | {len(df_gold_final.columns)} colunas")
df_gold_final.show(5, truncate=False)

## 📋 Tabela Gold

In [0]:
display(df_gold_final)

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(TABELA_GOLD):

    delta_table = DeltaTable.forName(spark, TABELA_GOLD)

    (
        delta_table.alias("target")
        .merge(
            df_gold_final.alias("source"),
            """
            target.coin_id = source.coin_id
            AND target.extraido_em = source.extraido_em
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"✅ Merge realizado com sucesso em {TABELA_GOLD}")

else:
    (
        df_gold_final.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TABELA_GOLD)
    )

    print(f"✅ Tabela criada: {TABELA_GOLD}")